# Milestone 3.1: Univariate, Correlation & Class-Balance Analysis

**Project**: AI-Driven Water Quality Prediction & Decision Support System  
**Milestone**: 3.1 (Exploratory Data Analysis)  
**Owner**: Data Lead (Person A) & ML Lead (Person B)  
**Dataset**: CPCB Surface Water Quality Monitoring Dataset (`data/interim/cpcb_labeled.csv`)  

---

## 1. Executive Summary & Objectives

This notebook performs the comprehensive Exploratory Data Analysis (EDA) mandated by the project specification:
1. **Univariate Distribution & Skewness Analysis**: Examine distribution shape, central tendencies, and heavy tails for all frozen numeric candidate features (`pH`, `DO`, `BOD`, `Fecal Coliform`).
2. **Correlation & Redundancy Analysis**: Calculate Pearson ($r$) and Spearman ($\rho$) correlation matrices to detect linear/monotonic relationships and flag pairs with $|r| > 0.85$ for redundancy review.
3. **Target Class Imbalance Audit**: Quantify multi-class (`wqi_class`) and binary (`Potability`) ground-truth class distributions to formally guide the resampling strategy (SMOTE / Class Weighting / Stratified K-Fold) in Phase 6.
4. **Written Interpretation**: Document explicit domain findings under every visualization to translate statistical properties into modeling requirements.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load_data import load_raw_data
from src.data.preprocessing import FROZEN_CANDIDATE_FEATURES
from src.features.eda import (
    compute_summary_statistics,
    compute_correlation_matrix,
    find_redundant_feature_pairs,
    compute_class_distribution,
)

# Visual formatting
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# Load dataset
df = load_raw_data(PROJECT_ROOT / "data/interim/cpcb_labeled.csv")
print(f"Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")

---  
## 2. Univariate Summary Statistics & Distribution Shapes

In [ ]:
summary_stats = compute_summary_statistics(df, FROZEN_CANDIDATE_FEATURES)
display_cols = ["feature", "missing_pct", "mean", "std", "min", "median", "max", "iqr", "skewness", "kurtosis"]
summary_stats[display_cols].style.format({
    "missing_pct": "{:.2f}%".format,
    "mean": "{:.2f}".format,
    "std": "{:.2f}".format,
    "min": "{:.2f}".format,
    "median": "{:.2f}".format,
    "max": "{:.2f}".format,
    "iqr": "{:.2f}".format,
    "skewness": "{:.2f}".format,
    "kurtosis": "{:.2f}".format,
}).set_caption("Univariate Summary Statistics for Candidate Water Quality Features")

### Univariate Distribution Visualizations (Histogram + Boxplot)

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(15, 18))

for i, col in enumerate(FROZEN_CANDIDATE_FEATURES):
    series = df[col].dropna()
    
    # Left: Histogram with KDE
    sns.histplot(series, kde=True, ax=axes[i, 0], color="#2b5c8f", bins=30)
    axes[i, 0].set_title(f"Histogram & KDE: {col}", fontsize=11, fontweight="bold")
    axes[i, 0].set_ylabel("Count")
    axes[i, 0].text(
        0.95, 0.90,
        f"Skew: {series.skew():.2f}\nMedian: {series.median():.2f}",
        transform=axes[i, 0].transAxes,
        va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8, edgecolor="gray"),
    )
    
    # Right: Box Plot
    sns.boxplot(x=series, ax=axes[i, 1], color="#4ca1af", fliersize=4)
    axes[i, 1].set_title(f"Box Plot: {col}", fontsize=11, fontweight="bold")
    
plt.tight_layout()
plt.show()

### **Univariate Interpretation & Modeling Implications**
1. **Potential of Hydrogen (pH)**: Symmetric, quasi-normal distribution centered around **7.70** (mean: 7.66, IQR: 7.30 – 8.10), ranging from 3.45 (acidic drain) to 9.20 (alkaline discharge). Skewness is **-0.47** with kurtosis 2.52. Standard scaling is optimal without transformation.
2. **Dissolved Oxygen (DO)**: Distribution centered at median **6.20 mg/L** (mean: 6.09 mg/L, IQR: 4.80 – 7.40 mg/L), ranging from 0.30 to 28.00 mg/L. Skewness is **-0.24** with kurtosis 4.95. Low values ($<2.0\text{ mg/L}$) capture acute hypoxic/eutrophic conditions, while supersaturated values ($>14.0\text{ mg/L}$) reflect algal photosynthetic surges.
3. **Biochemical Oxygen Demand (BOD)**: Severe positive skew (skewness: **+4.07**, kurtosis: 20.09) with median **2.80 mg/L** (mean: 6.94 mg/L) and upper tail reaching **127.0 mg/L**. Extreme values represent industrial/municipal sewage outfalls. Linear models will benefit from logarithmic or robust scaling.
4. **Fecal Coliform (FC)**: Extreme heavy-tailed distribution (skewness: **+33.98**, kurtosis: 1354.55) with median **140.0 MPN/100mL** (mean: 24,082.91 MPN/100mL) spanning from pristine baseline ($2.0\text{ MPN/100mL}$) to severe bacterial contamination ($2.2 \times 10^7\text{ MPN/100mL}$). Confirms that tree-based non-parametric algorithms (XGBoost, Random Forest, LightGBM) or robust scaling are naturally suited to this domain.

---  
## 3. Correlation & Multicollinearity Analysis

In [ ]:
pearson_corr = compute_correlation_matrix(df, method="pearson")
spearman_corr = compute_correlation_matrix(df, method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
mask = np.triu(np.ones_like(pearson_corr, dtype=bool))

# Pearson
sns.heatmap(pearson_corr, mask=mask, annot=True, fmt=".3f", cmap="coolwarm",
            vmin=-1.0, vmax=1.0, ax=axes[0], linewidths=0.5, cbar_kws={"label": "Pearson r"})
axes[0].set_title("Linear Feature Correlation (Pearson r)", fontsize=12, fontweight="bold")

# Spearman
sns.heatmap(spearman_corr, mask=mask, annot=True, fmt=".3f", cmap="coolwarm",
            vmin=-1.0, vmax=1.0, ax=axes[1], linewidths=0.5, cbar_kws={"label": "Spearman rho"})
axes[1].set_title("Monotonic Feature Correlation (Spearman Rank rho)", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Redundancy check against threshold |r| > 0.85
redundant_pairs = find_redundant_feature_pairs(pearson_corr, threshold=0.85)
print(f"Number of redundant feature pairs (|r| > 0.85): {len(redundant_pairs)}")
if redundant_pairs:
    print("Flagged pairs for review:", redundant_pairs)
else:
    print("Finding: No pairwise correlation exceeded the |r| > 0.85 redundancy threshold. No features are flagged as redundant.")

### **Correlation Findings & Redundancy Review**
- **DO vs BOD**: Exhibits a modest negative monotonic correlation ($\rho = -0.36$), physically reflecting organic consumption of dissolved oxygen during microbial decomposition.
- **BOD vs Fecal Coliform**: Shows a positive correlation ($r = +0.31$, $\rho = +0.51$), as raw domestic sewage discharge elevates both organic matter and coliform bacteria.
- **pH vs Others**: Displays very weak correlation with organic/biological parameters ($|r| < 0.12$), providing independent chemical equilibrium information.
- **Redundancy Decision**: **No pairwise correlation exceeded the |r| > 0.85 redundancy threshold**; all 4 candidate features are retained for modeling.

---  
## 4. Target Class Balance & Severity Audit

In [ ]:
class_dist = compute_class_distribution(df, target_column="wqi_class", output_csv=PROJECT_ROOT / "reports/class_distribution.csv")
class_dist["percentage_str"] = class_dist["percentage"].apply(lambda x: f"{x:.2f}%")
print("=== GROUND-TRUTH WQI CLASS DISTRIBUTION ===")
print(class_dist[["class", "count", "percentage_str"]].to_string(index=False))

# Binary Potability
pot_dist = compute_class_distribution(df, target_column="Potability", output_csv=None)
pot_dist["potability_label"] = pot_dist["class"].map({1: "Potable (Classes A & B)", 0: "Non-Potable (Classes C, D, E)"})
print("\n=== BINARY POTABILITY DISTRIBUTION ===")
print(pot_dist[["potability_label", "count", "percentage"]].to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 5.5))
palette = ["#2ecc71", "#3498db", "#f39c12", "#e67e22", "#e74c3c"]

ax = sns.barplot(
    data=class_dist,
    x="class",
    y="count",
    hue="class",
    palette=palette,
    legend=False,
    edgecolor="black",
    linewidth=0.8,
)

total = class_dist["count"].sum()
for i, row in class_dist.iterrows():
    cnt = int(row["count"])
    pct = float(row["percentage"])
    ax.text(
        i, cnt + (total * 0.015),
        f"{cnt}\n({pct:.2f}%)",
        ha="center", va="bottom",
        fontsize=10, fontweight="bold",
    )

plt.title("Ground-Truth WQI Class Distribution (CPCB Dataset)", fontsize=13, fontweight="bold")
plt.xlabel("Water Quality Class", fontsize=11)
plt.ylabel("Observation Count", fontsize=11)
plt.ylim(0, class_dist["count"].max() * 1.15)
plt.tight_layout()
plt.show()

### **Class Imbalance Severity & Strategic Recommendations for ML Track (Phase 5 & 6)**

| Class | Category | Count | Percentage | Imbalance Severity |
|:---|:---|---:|---:|:---|
| **Class A (Excellent)** | Minority | 7 | **0.21%** | **Extreme Minority** (Prone to zero-shot omission in standard splits) |
| **Class B (Good)** | Minority | 110 | **3.35%** | **Severe Minority** |
| **Class C (Poor)** | Intermediate | 455 | **13.84%** | Moderate |
| **Class D (Very Poor)** | Major | 919 | **27.96%** | High |
| **Class E (Unsuitable)** | Majority | 1,796 | **54.64%** | **Dominant Majority** |

#### **Critical Directives for Modeling:**
1. **Stratified Splitting**: `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` is mandatory in Milestone 4.2 to guarantee that Class A ($N=7$) is distributed representationally across training and validation folds.
2. **Evaluation Metrics**: Accuracy is an uninformative metric due to Class E ($54.64%$) dominance. **Macro F1-Score**, **Stratified Balanced Accuracy**, and per-class **Precision-Recall Curves** must serve as primary model comparison criteria.
3. **Resampling & Weighting (M6.1)**: Techniques such as class-weighted loss functions (`class_weight='balanced'`), focal loss, or adaptive oversampling (SMOTE / Borderline-SMOTE) must be benchmarked against baseline unweighted models.